<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 100px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Intelligence Data Science and Artificial Intelligence (IDEAI)</p>
  </div>

</div>

# **Hugging Face con fine-tuning real usando Trainer API**

## 1. Cambio de paradigma

En deep learning moderno muchas veces ya no entrenamos modelos desde cero.
Partimos de un modelo preentrenado y lo adaptamos a nuestro problema.

Eso se llama **fine-tuning**.

## 2. Por qué esto es tan importante

Fine-tuning permite:
- usar menos datos,
- entrenar más rápido,
- aprovechar conocimiento previo del modelo,
- y obtener resultados mucho mejores que con un entrenamiento desde cero en muchos casos.

## 3. En este notebook

Vamos a usar:
- `datasets`
- `transformers`
- `Trainer`
- `TrainingArguments`

Y haremos un fine-tuning real sobre un dataset de texto.

In [ ]:
# Si falta alguna librería, se puede instalar con:
# pip install transformers datasets accelerate evaluate

import numpy as np
import pandas as pd

## 4. Dataset de ejemplo

Usaremos el dataset **imdb**, un clásico de clasificación binaria de texto:
- positivo
- negativo

No es bancario, pero es perfecto para enseñar la mecánica del fine-tuning.
Después, en el caso final del día, trasladaremos la lógica a un entorno bancario.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

print(dataset)
print(dataset["train"][0])

## 5. Tokenización

Los modelos Transformers no entienden texto crudo directamente.
Antes hay que convertir el texto en tokens y tensores.

Usaremos un tokenizer asociado a DistilBERT.

In [ ]:
from transformers import AutoTokenizer

checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")

print(tokenized_datasets)

## 6. Modelo base preentrenado

Ahora cargamos el modelo para clasificación secuencial.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

## 7. Métricas

Vamos a definir accuracy y F1 para evaluar.

In [ ]:
import evaluate

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_metric.compute(predictions=preds, references=labels)
    f1 = f1_metric.compute(predictions=preds, references=labels)

    return {
        "accuracy": acc["accuracy"],
        "f1": f1["f1"]
    }

## 8. Trainer API

Esta es una de las piezas más importantes del notebook.
Aquí el alumno ve cómo se organiza un fine-tuning real.

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="results_imdb",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"].shuffle(seed=42).select(range(4000)),
    eval_dataset=tokenized_datasets["test"].shuffle(seed=42).select(range(1000)),
    compute_metrics=compute_metrics
)

## 9. Entrenamiento real

En clase conviene limitar el tamaño del dataset para que no se haga eterno.
Por eso aquí usamos una muestra reducida.

In [ ]:
trainer.train()

## 10. Evaluación

Después del entrenamiento, evaluamos.

In [ ]:
eval_results = trainer.evaluate()
eval_results

## 11. Predicción sobre frases nuevas

Esto es muy útil en clase porque convierte un entrenamiento abstracto en algo tangible.

In [ ]:
texts = [
    "The service was excellent and I am very satisfied.",
    "This was a terrible experience and I want a refund.",
    "The product is fine, but delivery was slow."
]

inputs = tokenizer(texts, return_tensors="pt", truncation=True, padding=True, max_length=256)
outputs = trainer.model(**inputs)
preds = outputs.logits.argmax(dim=-1).detach().cpu().numpy()

for text, pred in zip(texts, preds):
    print(text, "->", pred)

## 12. Qué es lo más importante aquí

No es solo ejecutar `Trainer`.
Lo importante es que el alumno entienda:

- qué significa partir de un checkpoint,
- por qué tokenizamos,
- qué hace `TrainingArguments`,
- qué papel cumple `Trainer`,
- y cómo evaluar el modelo resultante.

## 13. Traslado a banca

La misma lógica puede aplicarse a:
- emails de reclamación,
- incidencias de clientes,
- tickets internos,
- mensajes sospechosos,
- clasificación de texto de fraude,
- priorización automática de casos.

En banca, el modelo no sería imdb, pero el flujo técnico sería muy parecido.

## 14. Ejercicios propuestos

1. Cambia el checkpoint por otro modelo compatible.  
2. Aumenta `num_train_epochs` a 2 y compara.  
3. Reduce `max_length` y analiza el impacto.  
4. Cambia la métrica principal a accuracy.  
5. Entrena con otra muestra del dataset.  
6. Crea una función para predecir texto libre.  
7. Explica qué es fine-tuning con tus palabras.  
8. Compara entrenamiento desde cero vs modelo preentrenado.  
9. Propón un caso de clasificación textual en banca.  
10. Explica por qué Hugging Face cambia la práctica del NLP moderno.

In [ ]:
# Escribe aquí tu solución

## 15. Solucionario orientativo

- otro checkpoint posible: `bert-base-uncased`
- más épocas pueden mejorar, pero también sobreajustar
- truncation y max_length afectan coste y contexto
- fine-tuning = adaptar un modelo preentrenado a una tarea concreta
- en banca: clasificación de emails, incidencias, fraude, reclamaciones